# Construction Safety Monitor — Fire/Smoke Retraining **v2** (merged, smoke-focused)

Goal: push the weak **smoke** class up by (1) **merging several fire/smoke datasets** to fix the ~5:1 fire:smoke imbalance that capped smoke recall at ~0.55, (2) **oversampling smoke** images toward balance, and (3) training at **higher resolution (960px)** so thin/distant smoke is detectable.

### Setup (right-hand panel)
1. **Settings → Accelerator → GPU** — **P100** (or **T4 x2**; T4 is actually faster for AMP).
2. **Settings → Internet → On**.
3. **+ Add Input → DATASETS tab** (NOT Notebooks!) — add 2–3 fire/smoke datasets, e.g.:
   - `cubeai/fire-and-smoke-detection-for-yolov8`
   - `sayedgamal99/smoke-fire-detection-yolo`
   - `azimjaan21/fire-and-smoke-dataset-object-detection-yolo`
   The merge cell auto-detects whatever you add and remaps every class to `fire=0 / smoke=1` (handles Chinese 火/烟 and drops unrelated classes).

### How to run  ⚠️ read this
- From a **fresh session**, hit **Run All**.
- **Do NOT restart after the install cell** — on Kaggle a restart wipes the pip install and the P100 torch fix is lost. Install must run first, then everything else in the same session.

## 0. Install (P100-compatible torch) + GPU check
The P100 is Pascal (**sm_60**). The default ultralytics install pulls a CUDA-12.8 torch built only for sm_70+, which **cannot** run on a P100. We install a CUDA-12.4 torch (still ships sm_60) first, then ultralytics.

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q ultralytics roboflow

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print('torch:', torch.__version__, '| GPU:', gpu)
print('archs:', torch.cuda.get_arch_list())
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU'
if 'P100' in gpu:
    assert 'sm_60' in torch.cuda.get_arch_list(), \
        'WRONG TORCH for P100 - run from a FRESH session with Run All; do NOT restart after install'
print('>>> install OK - do NOT restart; just keep running cells <<<')

## 1. Discover mounted datasets
Lists what you actually attached. You must see real dataset paths here (not `/kaggle/input/notebooks/...`). If you only see a `notebooks` path, you added a notebook by mistake — add the **datasets** on the Datasets tab.

In [ ]:
import glob, os
print('--- mounted inputs ---')
for p in sorted(glob.glob('/kaggle/input/*')): print(' ', p)
ds_yamls = [y for y in glob.glob('/kaggle/input/**/data.yaml', recursive=True) if '/notebooks/' not in y]
print('--- dataset data.yaml files ---')
for y in ds_yamls: print(' ', y)
assert ds_yamls, 'No DATASET data.yaml found. Add fire/smoke datasets via + Add Input on the DATASETS tab (not Notebooks).'

## 2. Merge datasets → unified `['fire','smoke']` + oversample smoke
For every attached dataset this cell:
- reads its class names and builds a remap to `fire=0 / smoke=1` (handles `fire/flame/火`, `smoke/smog/烟`; drops anything else),
- **symlinks** images (no copy — keeps disk usage near zero) and rewrites label files with the remapped class IDs into `/kaggle/working/merged`,
- reports the fire:smoke instance balance and **oversamples smoke-containing train images** toward 1:1.

In [ ]:
import glob, os, re, shutil, yaml

MERGED = '/kaggle/working/merged'
EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def canon(name):
    n = str(name).strip().lower()
    if '火' in n or 'fire' in n or 'flame' in n: return 0
    if '烟' in n or 'smoke' in n or 'smog' in n: return 1
    return None

def split_of(path):
    p = path.lower()
    if 'val' in p:  return 'val'
    if 'test' in p: return 'test'
    return 'train'

ds_yamls = [y for y in glob.glob('/kaggle/input/**/data.yaml', recursive=True) if '/notebooks/' not in y]
assert ds_yamls, 'No dataset data.yaml under /kaggle/input - add datasets on the Datasets tab.'

for sp in ('train', 'val'):
    os.makedirs(f'{MERGED}/{sp}/images', exist_ok=True)
    os.makedirs(f'{MERGED}/{sp}/labels', exist_ok=True)

counts = {'train': {0: 0, 1: 0}, 'val': {0: 0, 1: 0}}
smoke_imgs = []

for y in ds_yamls:
    ds_root = os.path.dirname(y)
    tag = re.sub(r'[^A-Za-z0-9]+', '_', os.path.relpath(ds_root, '/kaggle/input'))[:40]
    with open(y) as f:
        d = yaml.safe_load(f) or {}
    names = d.get('names', [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    remap = {i: canon(nm) for i, nm in enumerate(names)}
    print(f'{ds_root}\n   names={names}  remap={remap}')

    for lbl in glob.glob(f'{ds_root}/**/labels/**/*.txt', recursive=True):
        sp = split_of(lbl)
        if sp == 'test':
            continue
        stem = os.path.splitext(os.path.basename(lbl))[0]
        img_dir = os.path.dirname(lbl).replace('/labels', '/images')
        img = None
        for e in EXTS:
            cand = os.path.join(img_dir, stem + e)
            if os.path.exists(cand):
                img = cand
                break
        if img is None:
            continue
        out, has_smoke = [], False
        with open(lbl) as f:
            for line in f:
                parts = line.split()
                if not parts:
                    continue
                new = remap.get(int(float(parts[0])))
                if new is None:
                    continue
                if new == 1:
                    has_smoke = True
                out.append(' '.join([str(new)] + parts[1:]))
                counts[sp][new] += 1
        if not out:
            continue
        name = f'{tag}__{stem}'
        dst_img = f'{MERGED}/{sp}/images/{name}{os.path.splitext(img)[1]}'
        if not os.path.exists(dst_img):
            os.symlink(os.path.realpath(img), dst_img)
        dst_lbl = f'{MERGED}/{sp}/labels/{name}.txt'
        with open(dst_lbl, 'w') as f:
            f.write('\n'.join(out) + '\n')
        if sp == 'train' and has_smoke:
            smoke_imgs.append((dst_img, dst_lbl))

print('\ninstances before oversampling:', counts)

f0, f1 = counts['train'][0], counts['train'][1]
if f1 and f0 > f1:
    factor = min(3, max(1, round(f0 / f1) - 1))
    print(f'train fire:smoke = {f0}:{f1} -> adding {factor} extra copy(ies) of smoke images')
    for k in range(factor):
        for img, lbl in smoke_imgs:
            b = os.path.splitext(os.path.basename(img))[0]
            ext = os.path.splitext(img)[1]
            di = f'{MERGED}/train/images/{b}_dup{k}{ext}'
            dl = f'{MERGED}/train/labels/{b}_dup{k}.txt'
            if not os.path.exists(di):
                os.symlink(os.path.realpath(img), di)
            shutil.copy(lbl, dl)

n_va = len(glob.glob(f'{MERGED}/val/images/*'))
if n_va == 0:
    print('No val split detected - carving 10% of train as val')
    for p in sorted(glob.glob(f'{MERGED}/train/images/*'))[::10]:
        b = os.path.basename(p); stem = os.path.splitext(b)[0]
        os.rename(p, f'{MERGED}/val/images/{b}')
        os.rename(f'{MERGED}/train/labels/{stem}.txt', f'{MERGED}/val/labels/{stem}.txt')

with open(f'{MERGED}/data.yaml', 'w') as f:
    yaml.safe_dump({'path': MERGED, 'train': 'train/images', 'val': 'val/images',
                    'nc': 2, 'names': ['fire', 'smoke']}, f)

n_tr = len(glob.glob(f'{MERGED}/train/images/*'))
n_va = len(glob.glob(f'{MERGED}/val/images/*'))
print(f'\nMERGED ready: {n_tr} train imgs, {n_va} val imgs')
print('data.yaml ->', f'{MERGED}/data.yaml')

## 3. Train (yolov8m, 960px, smoke-focused)
Higher resolution is the single best knob for smoke. Tune the config block in one place.

⚠️ **Time:** 960px + merged data is slow on a P100. Watch epoch-1 time — if it's >14 min/epoch you won't finish in Kaggle's 12h limit; drop `IMGSZ` to 768 or lower `EPOCHS`. To run unattended, use **Save Version → Save & Run All (Commit)**.

In [ ]:
from ultralytics import YOLO

IMGSZ  = 960          # smoke detail; drop to 768 if epochs are too slow
BATCH  = 8            # P100 16GB @ 960px; drop to 4 on CUDA OOM
EPOCHS = 50
MODEL  = 'yolov8m.pt' # try 'yolov8l.pt' for more capacity if time allows

model = YOLO(MODEL)
model.train(
    data='/kaggle/working/merged/data.yaml',
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, name='fire_smoke_v2', workers=4,
    cache=False, patience=15, save=True, val=True,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
    scale=0.6, translate=0.1, fliplr=0.5, degrees=5.0,
    mosaic=1.0, mixup=0.15, copy_paste=0.0,
    close_mosaic=10, cos_lr=True,
)
print('best ->', 'runs/detect/fire_smoke_v2/weights/best.pt')

## 4. Evaluate — watch the **smoke** row
`smoke mAP50` is the number this whole notebook exists to raise. Compare it to your old 0.667 baseline (keeping in mind val sets differ — section 5 does the apples-to-apples check).

In [ ]:
from ultralytics import YOLO
r = YOLO('runs/detect/fire_smoke_v2/weights/best.pt').val(data='/kaggle/working/merged/data.yaml', verbose=True)
ap = {int(c): float(a) for c, a in zip(r.box.ap_class_index, r.box.ap50)}
print('all   mAP50:', round(r.box.map50, 3))
print('fire  mAP50:', round(ap.get(0, float('nan')), 3))
print('smoke mAP50:', round(ap.get(1, float('nan')), 3))

## 5. (Optional) Fair comparison vs your current model
The only valid old-vs-new comparison runs **both models on the same val set**. Upload your current `models/firesmoke_model.pt` via **+ Add Input → Upload**, set `OLD` to its path, and run.

In [ ]:
import os
from ultralytics import YOLO

OLD = '/kaggle/input/old-firesmoke/firesmoke_model.pt'   # <-- edit to your uploaded path
NEW = 'runs/detect/fire_smoke_v2/weights/best.pt'
VAL = '/kaggle/working/merged/data.yaml'

def line(tag, w):
    r = YOLO(w).val(data=VAL, verbose=False)
    ap = {int(c): float(a) for c, a in zip(r.box.ap_class_index, r.box.ap50)}
    print(f'{tag:4s} | all {round(r.box.map50,3)} | fire {round(ap.get(0,float("nan")),3)} | smoke {round(ap.get(1,float("nan")),3)}')

if os.path.exists(OLD):
    line('OLD', OLD)
    line('NEW', NEW)
    print('Higher smoke on the SAME val set wins.')
else:
    print('Upload your current firesmoke_model.pt (+ Add Input > Upload) and set OLD to its path.')

## 6. Export weights
Copies `best.pt` to `/kaggle/working/firesmoke_model.pt` (already named for the backend). Download it from the **Output** panel, drop into your repo at `models/firesmoke_model.pt`, and restart `uvicorn` — class names are `fire`/`smoke`, so no code change.

In [ ]:
import shutil
shutil.copy('runs/detect/fire_smoke_v2/weights/best.pt', '/kaggle/working/firesmoke_model.pt')
print('Saved /kaggle/working/firesmoke_model.pt - download from the Output panel.')